# Sentinel Gate — Stage 4: export replay weeks + train ONE candidate — one dataset per run

**Set `FILE_TYPE` in cell 1** (start with `"Win64"`).

**What this does**
1. Loads the code bundle and installs the *locked* dependencies (same `uv.lock` as the local sandbox).
2. Re-downloads the official EMBER2024 feature records for `FILE_TYPE` (SHA-256 verified) and rebuilds the Stage 1 splits.
3. Checks the rebuild has the same content as the committed Stage 1 manifest (`docs/cleaning/<type>.json`); any difference stops the run.
4. Exports the cleaned **test weeks 53–64** + the never-break set → `sentinel_stage4_replay_<type>.zip` (for the local replay stream).
5. Retrains a **candidate** at T = end of week 52 (`configs/retrain.yaml`: policy `last_n_weeks`, 26 weeks; threshold and calibration on weeks 49–52; only labels known by T) → `sentinel_stage4_candidate_<type>.zip`.

Both zips land in `/kaggle/working` (= the notebook's Output).

**Safety:** feature records only; no binaries are downloaded or executed; `EMBER2024-capa` is refused.

**Kaggle settings:** Accelerator **None** (CPU only), Internet **On**, the code-bundle dataset attached. Then **Save Version → Save & Run All (Commit)**.

In [ ]:
# 1) Settings + platform + disk/RAM check
FILE_TYPE = "Win64"            # <-- start with "Win64"
REQUIRED_FREE_GB = 45           # zips + intermediate + final Parquet for one type

import os, shutil, subprocess, sys, zipfile, hashlib, glob, time, importlib.util
from pathlib import Path

assert FILE_TYPE in ("Win64", "Win32"), "FILE_TYPE must be Win64 or Win32"
# Kaggle images ship the google.colab package and a /content folder, and Colab images can have
# a /kaggle folder, so check Kaggle FIRST, by Kaggle's own markers.
ON_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
             or (Path("/kaggle/input").is_dir() and any(Path("/kaggle/input").iterdir())))
ON_COLAB = not ON_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ
                              or importlib.util.find_spec("google.colab") is not None)
WHERE = "Kaggle notebook" if ON_KAGGLE else ("Google Colab" if ON_COLAB else "other notebook host")

# Never put bulk data in /kaggle/working: Kaggle saves that folder as notebook output (20 GB cap).
candidates = [Path(p) for p in ["/content", "/kaggle/tmp", "/tmp"] if Path(p).exists()]
free = {p: shutil.disk_usage(p).free / 1e9 for p in candidates}
SCRATCH = max(free, key=free.get)
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else SCRATCH / "sentinel_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"Host: {WHERE} | FILE_TYPE: {FILE_TYPE} | scratch: {SCRATCH} ({free[SCRATCH]:.1f} GB free) | "
      f"outputs: {OUT_DIR} | CPUs: {os.cpu_count()} | RAM: {ram_gb:.1f} GB")
if free[SCRATCH] < REQUIRED_FREE_GB:
    raise SystemExit(f"Not enough disk: need {REQUIRED_FREE_GB} GB free. Tell Claude.")
if ram_gb < 16:
    raise SystemExit(f"Candidate training needs ~12 GB RAM; this machine has {ram_gb:.1f} GB. Tell Claude.")

CODE, DATA, VENV = SCRATCH / "sentinel_code", SCRATCH / "sentinel_data", SCRATCH / "sentinel_venv"
CAND_DIR = SCRATCH / f"candidate_{FILE_TYPE}"
(DATA / "raw").mkdir(parents=True, exist_ok=True)

In [ ]:
# 2) Load the code bundle (Kaggle: attached dataset; Colab: upload when asked)
BUNDLE_NAME = "sentinel_gate_code.zip"

def find_bundle():
    zips = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True) + glob.glob(f"/content/{BUNDLE_NAME}")
    if zips:
        return Path(zips[0]), None
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/BUNDLE_INFO.json", recursive=True)]
    return None, (dirs[0] if dirs else None)

bundle, extracted = find_bundle()
if bundle is None and extracted is None and ON_COLAB:
    from google.colab import files
    files.upload()
    bundle, extracted = find_bundle()
if bundle is None and extracted is None:
    raise SystemExit(f"{BUNDLE_NAME} not found. Kaggle: Add Input -> your code-bundle dataset.")

shutil.rmtree(CODE, ignore_errors=True)
if bundle is not None:
    print(f"bundle: {bundle}\nsha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}")
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(CODE)
else:
    print(f"using extracted bundle at {extracted}")
    shutil.copytree(extracted, CODE)
for need in ("scripts/export_replay.py", "scripts/retrain_candidate.py", "configs/retrain.yaml",
             f"docs/cleaning/{FILE_TYPE}.json"):
    if not (CODE / need).exists():
        raise SystemExit(f"Bundle has no {need}: it is the old bundle. Upload the Stage 4 dist/sentinel_gate_code.zip.")
print(open(CODE / "BUNDLE_INFO.json").read()[:200], "...")

In [ ]:
# 3) Install locked dependencies into an isolated Python 3.11 venv
def run(cmd, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), env=env, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if '"event": "clean_step"' not in line:
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"command failed with exit code {proc.returncode}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.6"])
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV), UV_PYTHON="3.11", UV_LINK_MODE="copy")
run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--no-install-project"], env=env, cwd=CODE)
PY = VENV / "bin" / "python"
RUN_ENV = dict(os.environ, PYTHONPATH=f"{CODE / 'src'}:{CODE}", SENTINEL_DATA_DIR=str(DATA),
               HF_HOME=str(DATA / ".hf-home"), PYTHONUNBUFFERED="1")
run([PY, "-c", "import sys, thrember, lightgbm, sklearn; print(sys.version.split()[0], 'lightgbm', lightgbm.__version__, 'sklearn', sklearn.__version__)"], env=RUN_ENV)

In [ ]:
# 4) Download -> rebuild splits -> check -> export test weeks -> retrain candidate
t0 = time.time()
run([PY, "scripts/download_ember.py", "--file-types", FILE_TYPE, "--dest", DATA / "raw"], env=RUN_ENV, cwd=CODE)
t1 = time.time()
run([PY, "scripts/clean_data.py", "--file-types", FILE_TYPE, "--raw-dir", DATA / "raw",
     "--splits-dir", DATA / "splits", "--clean-dir", DATA / "clean",
     "--manifest-dir", SCRATCH / "rebuilt_manifests", "--report", SCRATCH / "rebuilt_report.md",
     "--workers", str(os.cpu_count() or 1)], env=RUN_ENV, cwd=CODE)
for z in (DATA / "raw").glob(f"{FILE_TYPE}_*.zip"):
    z.unlink()                                  # free disk
shutil.rmtree(DATA / ".hf-home", ignore_errors=True)
t2 = time.time()

# 4a) replay data: the cleaned TEST weeks + never-break set (checked against docs/cleaning first)
replay_zip = OUT_DIR / f"sentinel_stage4_replay_{FILE_TYPE}.zip"
run([PY, "scripts/export_replay.py", "--file-type", FILE_TYPE, "--splits-dir", DATA / "splits",
     "--clean-dir", DATA / "clean", "--out", replay_zip], env=RUN_ENV, cwd=CODE)
t3 = time.time()

# 4b) candidate retrained at T = end of week 52 (configs/retrain.yaml)
shutil.rmtree(CAND_DIR, ignore_errors=True)
run([PY, "scripts/retrain_candidate.py", "--file-type", FILE_TYPE, "--splits-dir", DATA / "splits",
     "--clean-dir", DATA / "clean", "--out", CAND_DIR, "--threads", str(os.cpu_count() or 0)],
    env=RUN_ENV, cwd=CODE)
cand_zip = OUT_DIR / f"sentinel_stage4_candidate_{FILE_TYPE}.zip"
with zipfile.ZipFile(cand_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(CAND_DIR.iterdir()):
        zf.write(f, f.name)
t4 = time.time()

print(f"\n== {FILE_TYPE}: download {(t1 - t0) / 60:.1f} min, clean {(t2 - t1) / 60:.1f} min, "
      f"export {(t3 - t2) / 60:.1f} min, candidate {(t4 - t3) / 60:.1f} min")
for z in (replay_zip, cand_zip):
    print("result:", z, f"{z.stat().st_size / 1e6:.1f} MB")
if ON_COLAB:
    from google.colab import files
    files.download(str(replay_zip)); files.download(str(cand_zip))

In [ ]:
# 5) Summary (download BOTH zips from the Output tab and give their paths to Claude)
import json
m = json.load(open(CAND_DIR / "metrics.json"))["models"]["lightgbm"]
info = json.load(open(CAND_DIR / "run_info.json"))
c = m["calibration"]
print(f"candidate ({info['policy']}, T = week {info['windows']['as_of_week']}, fit weeks "
      f"{info['windows']['fit'][0]}-{info['windows']['fit'][-1]}, {info['counts']['fit']['rows']:,} rows): "
      f"threshold {m['threshold_raw']:.6f}; calibration weeks recall {c['recall']:.4f} at FPR {c['fpr']:.5f}; "
      f"never-break flags {m['never_break_flags']}")
print("\nKaggle: download", f"sentinel_stage4_replay_{FILE_TYPE}.zip", "and",
      f"sentinel_stage4_candidate_{FILE_TYPE}.zip", "from the Output tab.")